# Model Validation Framework

Evaluate a registered AI model with prediction history and analyst feedback.

In [ ]:
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_curve, auc
from clario360 import Client

MODEL_NAME = __import__('os').environ.get('CLARIO360_MODEL_NAME', 'cyber-sigma-evaluator')
LOOKBACK_DAYS = int(__import__('os').environ.get('CLARIO360_MODEL_LOOKBACK_DAYS', '30'))
sdk = Client.from_env()
model = sdk.ai.models.get_by_name(MODEL_NAME)
print(f"Evaluating {model.get('name', MODEL_NAME)}")


## Join predictions with feedback

This uses alert status as a simple ground-truth proxy for true-positive vs false-positive outcomes.

In [ ]:
predictions = sdk.ai.predictions.list(model_id=model.id, date_from=(datetime.utcnow() - timedelta(days=LOOKBACK_DAYS)).isoformat(), per_page=1000)
pred_df = predictions.to_dataframe()
alerts = sdk.cyber.alerts.list(date_from=(datetime.utcnow() - timedelta(days=LOOKBACK_DAYS)).isoformat(), per_page=1000)
alert_df = alerts.to_dataframe()
if pred_df.empty or alert_df.empty:
    raise RuntimeError('Not enough prediction or alert data to evaluate the model.')
merged = pred_df.merge(alert_df[['id', 'status']], left_on='entity_id', right_on='id', how='inner')
merged['y_true'] = (merged['status'] != 'false_positive').astype(int)
probability_column = 'confidence' if 'confidence' in merged.columns else 'output_confidence' if 'output_confidence' in merged.columns else None
if probability_column is None:
    raise RuntimeError('Prediction payload is missing confidence/output_confidence.')
merged['y_prob'] = merged[probability_column].astype(float)
merged['y_pred'] = (merged['y_prob'] >= 0.5).astype(int)
print(classification_report(merged['y_true'], merged['y_pred'], target_names=['False Positive', 'True Positive']))

## Metrics and promotion decision support

The charts below help determine whether a new shadow version has earned promotion.

In [ ]:
precision = precision_score(merged['y_true'], merged['y_pred'])
recall = recall_score(merged['y_true'], merged['y_pred'])
f1 = f1_score(merged['y_true'], merged['y_pred'])
cm = confusion_matrix(merged['y_true'], merged['y_pred'])
roc_x, roc_y, _ = roc_curve(merged['y_true'], merged['y_prob'])
roc_auc = auc(roc_x, roc_y)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Predicted FP', 'Predicted TP']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix')
axes[1].plot(roc_x, roc_y, label=f'ROC (AUC={roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random')
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
fig.tight_layout()
plt.show()
print({'precision': precision, 'recall': recall, 'f1': f1, 'auc': roc_auc})
if f1 >= 0.8:
    print('Model meets the notebook threshold for creating a shadow candidate.')
    candidate = sdk.ai.models.create_version(model.id, config={'threshold': 0.5, 'source': 'notebook'}, metrics={'precision': precision, 'recall': recall, 'f1': f1, 'auc': roc_auc}, lifecycle_stage='shadow')
    print(candidate)
